# Baseline notebook

Create model baseline and analysis for it

In [ ]:
import pandas as pd 
import numpy as np

pd.set_option('display.max_columns', 200)
np.random.seed(42)

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.head()

## Data preparation (From EDA)

In [ ]:
to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for column in to_category_columns:
    df[column] = df[column].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
#FIXME: take a look later
df["Churn"] = df["Churn"] == "Yes"
    
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.dropna()
df["TotalCharges"].isna().sum()

print("Data preparation is done")
df.shape

## Model building

In [ ]:
df.columns

In [ ]:
# Drop customerId column as never significant
df = df.drop(columns="customerID")
df.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

df_ohe = pd.get_dummies(df)
df_ohe.shape

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

rf = RandomForestClassifier(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(rf, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

### Metrics explained

It simpler to explain on one class. I will use where Churn is True

Class True (Churn is True):
- Precision 0.64 — when model predicts "churn", it's correct only 64% of the time
- Recall 0.48 — it catches only 48% of actual churners

Same logic for the not churned users

As we are trying to minimize missed by model churns (FN error that costs business big money). We should maximize recall on True class.

Despite model performs well on false classes (can classify not churning customers) it's perform nearly a coin flip predict on actually churned ones. Such conduct can be caused by class imbalance (roughly $\approx30%$% customers from data were churned)

## Data imbalance handling

Let's evaluate class imbalance handling methods

In [ ]:
churn_ratio = df_ohe.groupby("Churn")["Churn"].count()
churn_ratio

In [ ]:
print(f"Churned to not churned ratio is {churn_ratio.iloc[1] / churn_ratio.iloc[0]:.2f}")

### Model tuning (class_weight = 'balanced')

In [ ]:
# Simply change parameter in model
rf = RandomForestClassifier(random_state=42, class_weight="balanced")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(rf, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

Metrics do not change at all. Class weights do not bring significant result

### Using GBDT model

As GBDT models are robust to class imbalance I will perform analysis also

I will train a LightGBM as GBDT model with simple `is_unbalanced` parameter for handling data imbalance

In [ ]:
%pip install lightgbm -q

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(is_unbalance=True, verbose=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(lgbm, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

GBDT with native data imbalance handling parameter performs much better than random forest. We gain 75% recall on our churned customers that is a good result for almost not tuned model

### Undersampling

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

pipeline = Pipeline([
    ("undersample", RandomUnderSampler(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(pipeline, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

Undersampled metrics are quite impressive. We achieve score like a GBDT model but using baseline. Undersample have decrease metrics on False classes (Model do more FP mistakes, thinking that customer will churn, while he will not). For our case this is acceptable (we are focusing on detecting True classes)

### Oversampling

#### Boostrap oversampling

Oversampling that will simply boostrap minority class

In [ ]:
from imblearn.over_sampling import RandomOverSampler

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

pipeline = Pipeline([
    ("oversample", RandomOverSampler(random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(pipeline, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

Boostrap oversampling improves our metrics but not that much as undersampling

#### SMOTE (SMOTENC) oversampling

Interpolate points to balance minority class

SMOTENC is a modification that handles mixed numerical and categorical features natively, so it works directly on the original dataframe

In [ ]:
from imblearn.over_sampling import SMOTENC

X = df.drop(columns="Churn").copy()
y = df["Churn"].copy()

cat_cols = X.select_dtypes(include=["category", "bool"]).columns.tolist()

# Convert categories to integers for SMOTENC
for col in cat_cols:
    X[col] = X[col].cat.codes if X[col].dtype.name == "category" else X[col].astype(int)

pipeline = Pipeline([
    ("smotenc", SMOTENC(categorical_features=cat_cols, random_state=42)),
    ("rf", RandomForestClassifier(random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(pipeline, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

SMOTE give us metrics similar to bootstrapping oversample. The cause might be that the data is mostly categorial with only 3 numeric features. Interpolation between points mostly duplicates categorical features causing bootstrapping with noise in numeric features

### GBDT + undersampling

In [ ]:
X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

pipeline = Pipeline([
    ("undersample", RandomUnderSampler(random_state=42)),
    ("lgb", lgb.LGBMClassifier(is_unbalance=True, verbose=-1))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(pipeline, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

GBDT with undersampling slightly improved our recall and decrease precision

## Conclusion

Plot of experiments for better analysis experience

In [ ]:
import matplotlib.pyplot as plt

models = [
    "RF\nBaseline",
    "RF\nBalanced",
    "LightGBM\n(is_unbalance)",
    "Undersampling\n+ RF",
    "Bootstrap\nOversampling + RF",
    "SMOTE\n+ RF",
    "GBDT\n+ Undersampling",
]

true_recall = [0.48, 0.48, 0.75, 0.76, 0.58, 0.58, 0.77]
true_precision = [0.64, 0.64, 0.53, 0.51, 0.59, 0.58, 0.50]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width / 2, true_precision, width, label="True Precision", color="#4C72B0")
bars2 = ax.bar(x + width / 2, true_recall, width, label="True Recall", color="#DD8452")

ax.set_ylabel("Score")
ax.set_title("True Class (Churn) — Precision & Recall by Model")
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


From plot we see that best approaches are using GBDT model with imbalance param or using an udersample technic

But all our manipulations with data decrease model precision from 0.68 to ~0.5. That means if model predict 10 churned people 5 will be actually churned. That leads to big potential losses on FP errors ($38$) and should be threated with PR-ROC curve analysis

## Recomendations

- Use an GBDT approach with class balance parameter
    - Or use an undersampling approach with Random Forest
- Analyze PR-ROC curve and manage best threshhold for predictions